In [47]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import string
import numpy as np

In [48]:
with open('persuasion.txt') as f:
    content = f.read()
print(len(content))

484132


In [49]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [50]:
allowed_chars = set(string.ascii_letters + string.digits + ". ")

content = ''.join([c for c in content if c in allowed_chars])

chars = set(sorted(list(content)))
i_to_c = {k:v for k, v in enumerate(chars)}
c_to_i = {k:v for v, k in i_to_c.items()}

encode = lambda s: [c_to_i[c] for c in s] # noqa: E731
decode = lambda s: ''.join([i_to_c[i] for i in s]) # noqa: E731
decode(encode(content[:100]))

encoded_content = torch.tensor(encode(content))

In [51]:
blocks = []
block_length = 6
batch_size = 32

xs = []
ys = []

for i in range(0, len(encoded_content)-block_length):
    x_chunk = encoded_content[i:i+block_length-1]
    y_chunk = encoded_content[i+1:i+block_length]

    xs.append(torch.tensor(x_chunk))
    ys.append(torch.tensor(y_chunk))

X = F.one_hot(torch.stack(xs)).float()
Y = torch.stack(ys)


/tmp/ipykernel_32343/3824314043.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  xs.append(torch.tensor(x_chunk))
/tmp/ipykernel_32343/3824314043.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ys.append(torch.tensor(y_chunk))


In [52]:
X = X.to(device)
Y = Y.to(device)

In [53]:

# x_input @ W +b1
hidden_layer_width = 512
relu = torch.nn.ReLU()

W1 = torch.randn(len(chars), hidden_layer_width, device=device)*0.01
W1.requires_grad_(True)

b1 = torch.zeros(hidden_layer_width, requires_grad=True, device=device)

W2 = torch.randn(hidden_layer_width, len(chars), device=device)*0.01
W2.requires_grad_(True)

b2 = torch.zeros(len(chars), requires_grad=True, device=device)


In [57]:
import time

timings = []
for epoch in range(1):

    for i in range(0, len(X), batch_size):
        
        torch.cuda.synchronize()
        start = time.perf_counter()
        indices = torch.randperm(len(X))
        indices = indices.to(device)
        batch_idx = indices[i:i+batch_size]
        

        x = X[batch_idx]
        y = Y[batch_idx]
        z1 = x @ W1 + b1
        h1 = relu(z1)
        logits = h1 @ W2 + b2
        loss = F.cross_entropy(logits.view(-1, len(chars)), y.view(-1))


        if W1.grad is not None:
            for p in [W1, W2, b1, b2]: 
                p.grad.zero_()

        loss.backward()
    

        with torch.no_grad():
            for p in [W1, W2, b1, b2]:
                p -= 0.01 * p.grad
        
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        if not i%1000:
            print(f'epoch {epoch}, loss {loss}, {i} of {len(X)} iterations')
            print(f'time for a single iter: {elapsed:.6f}')
            timings.append(elapsed)
            print(f'average time per iter so far: {np.mean(timings)}')

epoch 0, loss 4.158646106719971, 0 of 463240 iterations
time for a single iter: 0.785680
average time per iter so far: 0.7856802450005489
epoch 0, loss 4.087319374084473, 4000 of 463240 iterations
time for a single iter: 0.010231
average time per iter so far: 0.39795562800009066
epoch 0, loss 4.010141849517822, 8000 of 463240 iterations
time for a single iter: 0.012302
average time per iter so far: 0.2694044826666868
epoch 0, loss 3.9586119651794434, 12000 of 463240 iterations
time for a single iter: 0.012032
average time per iter so far: 0.20506144549995042
epoch 0, loss 3.894644260406494, 16000 of 463240 iterations
time for a single iter: 0.027793
average time per iter so far: 0.16960773320006411
epoch 0, loss 3.8349709510803223, 20000 of 463240 iterations
time for a single iter: 0.010457
average time per iter so far: 0.14308266683337934
epoch 0, loss 3.743415355682373, 24000 of 463240 iterations
time for a single iter: 0.012705
average time per iter so far: 0.12445732528574029
epoch

KeyboardInterrupt: 

In [58]:
inputs = 'h'

for i in range(100):
    x = F.one_hot(torch.tensor(encode(inputs[-1]), device=device), num_classes=len(chars))

    z1 = x.float() @ W1 + b1
    relu = torch.nn.ReLU()
    h = relu(z1)
    logits = h @ W2 + b2
    probs = F.softmax(logits, dim=-1)
    idx = torch.multinomial(probs, num_samples=1).item()
    inputs+=i_to_c[idx]
    # output = decode(logits.argmax(dim=1).tolist())

    # inputs+=output
    print(inputs)



h 
h y
h yh
h yhf
h yhfq
h yhfqn
h yhfqno
h yhfqno 
h yhfqno t
h yhfqno tu
h yhfqno tun
h yhfqno tuna
h yhfqno tunan
h yhfqno tunane
h yhfqno tunaner
h yhfqno tunaner 
h yhfqno tunaner  
h yhfqno tunaner  s
h yhfqno tunaner  sd
h yhfqno tunaner  sdc
h yhfqno tunaner  sdcq
h yhfqno tunaner  sdcqf
h yhfqno tunaner  sdcqfn
h yhfqno tunaner  sdcqfns
h yhfqno tunaner  sdcqfns 
h yhfqno tunaner  sdcqfns n
h yhfqno tunaner  sdcqfns ne
h yhfqno tunaner  sdcqfns nen
h yhfqno tunaner  sdcqfns neno
h yhfqno tunaner  sdcqfns neno 
h yhfqno tunaner  sdcqfns neno n
h yhfqno tunaner  sdcqfns neno nl
h yhfqno tunaner  sdcqfns neno nlr
h yhfqno tunaner  sdcqfns neno nlr 
h yhfqno tunaner  sdcqfns neno nlr  
h yhfqno tunaner  sdcqfns neno nlr  o
h yhfqno tunaner  sdcqfns neno nlr  oe
h yhfqno tunaner  sdcqfns neno nlr  oe8
h yhfqno tunaner  sdcqfns neno nlr  oe8a
h yhfqno tunaner  sdcqfns neno nlr  oe8aQ
h yhfqno tunaner  sdcqfns neno nlr  oe8aQh
h yhfqno tunaner  sdcqfns neno nlr  oe8aQh8
h yhfqno tuna